# Subset cuentas 6 y 7 (n=283): entrenamiento, resultados y reporte top-K

Notebook autocontenido: corre las celdas en orden. Las primeras entrenan
robust_z / AE denso / LSTM-AE sobre el subset; las siguientes muestran los
resultados comparativos; las ultimas generan el reporte top-K de candidatos
(usando `reporting.py`) para el anexo.

## 1. Carga y preparacion de datos

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from db import get_connection
from features_l2 import FeatureBuilder, FeatureConfig
from autoencoder import AnomalyAE, AEConfig
from lstm_autoencoder import LSTMAnomalyAE, LSTMConfig, make_lstm_scorer
from baseline import robust_zscore, MATERIALITY_USD
from eval_injection import evaluate, InjectionConfig
from reporting import candidates_table, export_candidates, to_markdown

In [ ]:
query = """
SELECT *
FROM notif.vw_ML_L2_CompleteSeries
WHERE LEFT(CAST(account AS varchar(20)), 1) IN ('6', '7')
"""

conn = get_connection(env="PROD")
df_clean = pd.read_sql(query, conn)
df_clean["period_date"] = pd.to_datetime(df_clean["period_date"])

print(f"Filas cargadas: {len(df_clean)}")
print(f"Cuentas distintas: {df_clean['account'].nunique()}")

In [ ]:
periodos_por_cuenta = df_clean.groupby("account").size()
UMBRAL_LSTM = 13  # W=12 + al menos 1 periodo para puntuar

cuentas_comunes = periodos_por_cuenta[periodos_por_cuenta >= UMBRAL_LSTM].index
n_cuentas = len(cuentas_comunes)
print(f"Cuentas comunes a los 3 modelos: {n_cuentas}")
assert n_cuentas == 283, f"Se esperaban 283 y se obtuvieron {n_cuentas}; revisa si los datos cambiaron."

cuentas_comunes_tuples = {(acct,) for acct in cuentas_comunes}
df_final = df_clean[df_clean["account"].isin(cuentas_comunes)].copy()

## 2. Split temporal (ultimos 6 meses de test) + FeatureBuilder

In [ ]:
MESES_TEST = 6
ultimo_periodo = df_final["period_date"].max()
split = ultimo_periodo - pd.DateOffset(months=MESES_TEST)
print(f"Fecha de corte train/test: {split} (ultimos {MESES_TEST} meses como test)")

feat_cfg = FeatureConfig()
fb = FeatureBuilder(feat_cfg)
fb.fit(df_final[df_final["period_date"] < split], selected_series=cuentas_comunes_tuples)

X_all = fb.transform(df_final)
X_train = X_all[X_all["period_date"] < split]
X_test = X_all[X_all["period_date"] >= split]

print(f"Cuentas tras fit/transform: {X_all['account'].nunique()}")
print(f"Train: {len(X_train)} filas | Test: {len(X_test)} filas")

## 3. Entrenar AE denso y LSTM-AE

In [ ]:
ae = AnomalyAE(
    feature_cols=fb.feature_cols,
    id_cols=["account", "period_name", "period_date", "net_movement"],
    cfg=AEConfig(),
)
ae.fit(X_train)

In [ ]:
lstm_ae = LSTMAnomalyAE(
    feature_cols=fb.feature_cols,
    series_keys=["account"],
    period_date_col="period_date",
    id_cols=["account", "period_name", "period_date", "net_movement"],
    cfg=LSTMConfig(),
)
lstm_ae.fit(X_train)

## 4. Evaluacion comparativa con eval_injection

In [ ]:
def scorer_robust_z(df):
    scored = robust_zscore(df, min_periods=12, materiality=MATERIALITY_USD)
    if scored.empty:
        return pd.DataFrame(columns=["account", "period_name", "score"])
    out = scored[["account", "period_name"]].copy()
    out["score"] = scored["abs_score_z"]
    return out

def scorer_ae(df):
    Xt = fb.transform(df)
    scored = ae.score_frame(Xt)
    return scored.rename(columns={"recon_error": "score"})[["account", "period_name", "score"]]

scorer_lstm = make_lstm_scorer(fb, lstm_ae, series_keys=["account"], period_col="period_name")

In [ ]:
cfg = InjectionConfig(n_injections=60, n_repeats=5)

res_robust_z = evaluate(df_final, scorer_robust_z, cfg, series_keys=["account"], detector_name="robust_z")
res_ae = evaluate(df_final, scorer_ae, cfg, series_keys=["account"], detector_name="AE_denso")
res_lstm = evaluate(df_final, scorer_lstm, cfg, series_keys=["account"], detector_name="LSTM-AE")

resultados = pd.concat([res_robust_z, res_ae, res_lstm], ignore_index=True)
resultados.to_csv("resultados_subset_cuentas_6_7.csv", index=False)
resultados

## 5. Tablas y graficos comparativos

In [ ]:
tabla_auc = resultados.pivot(index="inj_type", columns="detector", values="roc_auc_mean")
tabla_auc.round(3)

In [ ]:
tabla_ap = resultados.pivot(index="inj_type", columns="detector", values="ap_mean")
tabla_ap.round(3)

In [ ]:
tabla_recall = resultados.pivot(index="inj_type", columns="detector", values="recall_mean")
tabla_recall.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(tabla_auc, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=1.0,
           cbar_kws={"label": "ROC-AUC"})
plt.title("ROC-AUC por detector y tipo de anomalia (cuentas 6 y 7, n=283)")
plt.ylabel("Tipo de anomalia")
plt.xlabel("Detector")
plt.tight_layout()
plt.savefig("heatmap_auc_subset_cuentas_6_7.png", dpi=150)
plt.show()

In [ ]:
tabla_auc.plot(kind="bar", figsize=(10, 5))
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Azar (AUC=0.5)")
plt.title("Comparacion de detectores por tipo de anomalia (cuentas 6 y 7)")
plt.ylabel("ROC-AUC")
plt.xlabel("Tipo de anomalia")
plt.xticks(rotation=0)
plt.legend(title="Detector")
plt.tight_layout()
plt.savefig("barras_auc_subset_cuentas_6_7.png", dpi=150)
plt.show()

## 6. Reporte top-K de candidatos (LSTM-AE) para el anexo

In [ ]:
report = lstm_ae.score_frame(X_all)   # with_channels=True por defecto -> incluye anomaly_family
candidates_table(report, k=10)

In [ ]:
paths = export_candidates(
    report,
    out_dir="reports/subset_cuentas_6_7",
    k=10,
    prefix="candidatos_L2_subset67",
)
paths

In [ ]:
print(to_markdown(report, k=10))